
Think of panel data as “many people (or firms) observed over time.” FE, RE, and Pooled are three ways of handling the fact that each person/firm has its own hidden characteristics.

## Pooled regression

Pooled regression treats the data as if it were one big cross‑section and ignores who is who.  
- It acts as if all firms are identical except for the observed variables in the regression.  
- Any systematic differences between firms (management quality, technology, risk culture) are simply treated as random noise.  

In layman terms:  
“You shake all firm–year observations into one big bucket and fit a single line, pretending there are no stable differences between firms beyond what you see in the Xs.”

This is simple but often unrealistic with firm or country data, because entities do differ in persistent, unobserved ways.

## Fixed Effects (FE)

Fixed Effects explicitly allows each firm (or person, or country) to have its **own baseline level** that does not change over time.  
- It is like giving every firm its own intercept (its own starting point).  
- FE then explains how **changes over time within the same firm** in the X variables are associated with changes in Y.  
- Any time‑invariant firm characteristics (e.g., industry niche, corporate culture) are “controlled for” automatically, even if you never measure them.

In layman terms:  
“First, adjust for the fact that each firm has its own personality and average level. Then ask: when this particular firm’s value or capital goes up or down over time, how does its own investment respond?”

FE is the workhorse when you worry that unobserved, stable differences between firms are correlated with your regressors.

## Random Effects (RE)

Random Effects also recognizes that firms have different baseline levels, but it treats these differences as **random draws** from some distribution and assumes they are **not correlated** with the regressors.  
- It keeps more variation (both across firms and over time) than FE, so estimates are more efficient if the assumption holds.  
- But if unobserved firm traits are related to the X variables, RE becomes biased.

In layman terms:  
“We admit firms differ, but we assume those differences are unrelated to the explanatory variables, like random noise around a grand mean. Under that assumption, we can use both the between‑firm and within‑firm variation to estimate the effect.”

## How they differ conceptually

- **Pooled**: “Everyone is the same; only Xs matter.”  
- **FE**: “Everyone has their own level; only within‑entity changes over time identify effects; unobserved traits may be correlated with Xs.”  
- **RE**: “Everyone has their own random level; we assume those unobserved traits are not systematically related to Xs; we use both across and within variation.”

In practice:
- If the F‑test says firm effects matter, pooled is too naive.  
- If the Hausman test rejects RE, FE is safer because it does not rely on the “no correlation with unobserved traits” assumption.

The Grunfeld data set is like a long‑running “financial diary” of a small group of big manufacturing companies.[1][2][3]

## What the data contain

- You follow about **10–11 large US manufacturing firms** (like General Motors, US Steel, IBM, etc.)[2][1]
- For roughly **20 years in a row** (from **1935 to 1954**).[4][3][1]

For each firm and year, you see three main numbers:

- **Investment**: how much the firm spent on factories and equipment in that year (in millions of dollars, adjusted for inflation).[1][2]
- **Value**: the overall **market value** of the firm (stock market value of equity plus debt, again in comparable dollars).[2][1]
- **Capital**: the **existing stock of plant and equipment** the firm already has, measured in real terms.[1][2]

Plus two identifiers:

- **Firm**: which company this row belongs to (GM, US Steel, etc.).[3][1]
- **Year**: the calendar year of the observation.[3][1]

## Why it’s useful in teaching

In layman terms, the data let you ask:

- “For the same firm over time, when its value goes up or it accumulates more capital, does it invest more in new plants and equipment?”  
- “Do some firms inherently invest more than others, even after controlling for value and capital?”  

Because it tracks the **same firms over many years**, it is a textbook example for:

- **Pooled regression**: treat all firm–year points as one big sample.  
- **Fixed Effects**: control for each firm’s inherent “style” or baseline investment tendency.  
- **Random Effects**: assume firm‑specific traits are random and uncorrelated with value and capital.

So in simple language: Grunfeld is a classic panel of “big firms’ investment behavior over time” used to study how investment responds to firm size and existing capital, while controlling for persistent differences between firms.[2][3][1]

[1](https://search.r-project.org/CRAN/refmans/AER/html/Grunfeld.html)
[2](https://www.statsmodels.org/dev/datasets/generated/grunfeld.html)
[3](https://www.stat.auckland.ac.nz/~wild/data/Rdatasets/doc/plm/Grunfeld.html)
[4](https://tilburgsciencehub.com/topics/analyze/causal-inference/panel-data/paneldata/)
[5](https://www.statsmodels.org/0.6.1/datasets/generated/grunfeld.html)
[6](https://www.zeileis.org/papers/Kleiber+Zeileis-2010.pdf)
[7](https://www.scribd.com/document/364621782/Grunfeld)
[8](https://vincentarelbundock.github.io/Rdatasets/doc/plm/Grunfeld.html)
[9](https://algoritmaonline.com/regression-with-panel-data/)
[10](https://alexmarsella.github.io/R/panel_tutorial/)
[11](https://icar-iirr.org/books/chapters/Statistical-Procedures_ch9.pdf)
[12](https://github.com/statsmodels/statsmodels/blob/master/statsmodels/datasets/grunfeld/data.py)
[13](https://bookdown.org/ccolonescu/RPoE4/panel-data-models.html)
[14](https://pages.stern.nyu.edu/~wgreene/Econometrics/PanelDataSets.htm)
[15](https://www.rdocumentation.org/packages/systemfit/versions/1.1-30/topics/GrunfeldGreene)
[16](https://rdrr.io/cran/plm/man/Grunfeld.html)
[17](https://rdrr.io/rforge/plm/man/Grunfeld.html)
[18](https://search.r-project.org/CRAN/refmans/plm/html/Grunfeld.html)
[19](http://cob.uum.edu.my/images/PG_Unit_COB/SLIDE-event/Time_series_and_Panel_Data_tal_series_OYA_12_Julai_2021_part_2.pdf)
[20](https://mpra.ub.uni-muenchen.de/20841/1/MPRA_paper_20841.pdf)

In [41]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PooledOLS, PanelOLS, RandomEffects
from scipy.stats import chi2

# ========= 0. LOAD DATA =========
df = pd.read_csv("grunfeld.csv")
print("Columns:", df.columns.tolist())
# Adjust these names if your file uses different ones
# Expected: firm, year, invest, value, capital

df['firm'] = df['firm'].astype('category')
df = df.set_index(['firm', 'year']).sort_index()

y = df['invest']
X = df[['value', 'capital']]
X_const = sm.add_constant(X)

# ========= 1. ESTIMATE MODELS =========

# 1.1 Pooled OLS
pooled_model = PooledOLS(y, X_const)
pooled_res = pooled_model.fit()

# 1.2 Fixed Effects (within)
fe_model = PanelOLS(y, X, entity_effects=True)
fe_res = fe_model.fit()

# 1.3 Random Effects
re_model = RandomEffects(y, X_const)
re_res = re_model.fit()

print("\n=== MODELS ESTIMATED ===")
print("Pooled OLS, Fixed Effects, Random Effects.\n")

print("=== Pooled OLS ===")
print(pooled_res.summary)

print("\n=== Fixed Effects (FE) ===")
print(fe_res.summary)

print("\n=== Random Effects (RE) ===")
print(re_res.summary)

# ========= 2. TEST: POOLED VS FE (F-test for entity effects) =========
print("\n=== Test: Are firm (entity) effects jointly zero? (Pooled vs FE) ===")
fstat = fe_res.f_statistic
print(f"F-statistic: {fstat.stat:.4f}, p-value: {fstat.pval:.4f}")

print("\nInterpretation for Pooled OLS:")
if fstat.pval < 0.05:
    print("- The p-value is below 0.05.")
    print("- Firm-specific (entity) effects are statistically important.")
    print("- This contradicts the pooled OLS assumption that there are no relevant unobserved firm effects.")
    print("→ Pooled OLS is likely misspecified; FE or RE is preferred.")
else:
    print("- The p-value is above 0.05.")
    print("- There is no strong evidence that firm-specific effects matter.")
    print("- The pooled OLS assumption of no relevant unobserved firm effects is not rejected.")
    print("→ On this dimension, Pooled OLS may be acceptable (still check other assumptions).")

fstat = fe_res.f_statistic
print("=== F-test for firm (entity) effects ===")
print(f"F-statistic: {fstat.stat:.4f}")
print("\nInterpretation:")
if fstat.pval < 0.05:
    print("- p-value < 0.05 → reject the null that all firm effects are zero.")
    print("- This means unobserved firm-specific effects are statistically important.")
    print("- The pooled OLS assumption of 'no relevant firm effects' is violated.")
    print("→ Do NOT rely on pooled OLS; use a panel model with firm effects (FE or RE).")
else:
    print("- p-value ≥ 0.05 → cannot reject the null that firm effects are zero.")
    print("- There is no strong evidence that unobserved firm heterogeneity matters.")
    print("- On this dimension, pooled OLS is not rejected in favor of FE.")
    print("→ Pooled OLS may be acceptable (subject to other assumptions).")


# ========= 3. HAUSMAN TEST: FE VS RE =========
print("\n=== Hausman Test: Fixed Effects vs Random Effects ===")

# Take common coefficients: value, capital (no intercept)
b_fe = fe_res.params[['value', 'capital']]
b_re = re_res.params[['value', 'capital']]

V_fe = fe_res.cov.loc[b_fe.index, b_fe.index]
V_re = re_res.cov.loc[b_re.index, b_re.index]

diff = b_fe - b_re
V_diff = V_fe - V_re

H = float(diff.T @ np.linalg.inv(V_diff.values) @ diff.values)
df_h = diff.shape[0]
p_hausman = 1 - chi2.cdf(H, df_h)

print(f"Hausman statistic: {H:.4f}, df = {df_h}, p-value: {p_hausman:.4f}")

print("\nInterpretation for FE vs RE:")
if p_hausman < 0.05:
    print("- The p-value is below 0.05.")
    print("- FE and RE coefficients differ significantly.")
    print("- This suggests the regressors are correlated with unobserved firm effects.")
    print("→ The Random Effects exogeneity assumption fails; FE is preferred for consistent estimates.")
else:
    print("- The p-value is above 0.05.")
    print("- There is no strong statistical difference between FE and RE coefficients.")
    print("- This means we do not find evidence against the RE exogeneity assumption.")
    print("→ You may use Random Effects (more efficient than FE if its assumptions hold).")

# ========= 4. RESIDUAL DIAGNOSTICS (HETEROSKEDASTICITY) =========
print("\n=== Residual diagnostics: Heteroskedasticity across firms (FE model) ===")
fe_resid = fe_res.resids
firm_var = fe_resid.groupby(level='firm').var()
print("Residual variance by firm (first 10):")
print(firm_var.head(10))
print(f"Min residual variance across firms: {firm_var.min():.4f}")
print(f"Max residual variance across firms: {firm_var.max():.4f}")

print("\nInterpretation for error variance:")
print("- If residual variances differ a lot across firms, homoskedasticity is doubtful.")
print("- In practice, you should report robust or cluster-robust standard errors (e.g., by firm).")

# Optional: FE with cluster-robust SEs
fe_robust = PanelOLS(y, X, entity_effects=True).fit(cov_type='clustered', cluster_entity=True)
print("\n=== Fixed Effects with Cluster-Robust SEs ===")
print(fe_robust.summary)

print("\nInterpretation for robust SEs:")
print("- Cluster-robust SEs relax the assumptions of constant variance and no serial correlation within firms.")
print("- If robust SEs differ a lot from classical SEs, it indicates that classical error assumptions were unrealistic.")


Columns: ['invest', 'value', 'capital', 'firm', 'year']

=== MODELS ESTIMATED ===
Pooled OLS, Fixed Effects, Random Effects.

=== Pooled OLS ===
                          PooledOLS Estimation Summary                          
Dep. Variable:                 invest   R-squared:                        0.8179
Estimator:                  PooledOLS   R-squared (Between):              0.8426
No. Observations:                 220   R-squared (Within):               0.7357
Date:                Mon, Dec 15 2025   R-squared (Overall):              0.8179
Time:                        20:57:39   Log-likelihood                   -1301.3
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      487.28
Entities:                          11   P-value                           0.0000
Avg Obs:                       20.000   Distribution:                   F(2,217)
Min Obs:                       20.000        

The Grunfeld results can be read as a story about how big firms decide how much to invest each year, and whether it is legitimate to compare firms directly or only compare each firm to itself over time.[1][2][3]

## What the coefficients usually say

In all three models (Pooled, FE, RE), you are regressing **investment** on **firm value** and **existing capital stock**.[2][1]

- The coefficient on **value** is typically **positive**:  
  In plain language: “When a firm becomes more valuable in the market (its overall size/market valuation increases), it tends to invest more in plants and equipment.”  
- The coefficient on **capital** is also usually **positive but smaller**:  
  “Firms that already have more capital also tend to invest more, but the effect per unit of capital is smaller than the effect of value.”

So, the basic economic message is: **larger, more valuable firms invest more**, and firms with a bigger capital base also continue to invest, but at a somewhat lower marginal rate.[1][2]

## What the Fixed Effects vs Pooled tells you

From your F‑test, you saw that **firm effects are significant**, so pooled OLS is misspecified even though it had the highest R². In lay terms:

- Different firms have their own **inbuilt investment style** or “personality” (some are chronically high investors, some are cautious), beyond what value and capital can explain.  
- The data clearly show these firm-specific tendencies matter.  
- Pooled regression wrongly **pretends all firms are the same**, so it attributes too much to value and capital, inflating R².

The **Fixed Effects** model says:  
- “Let each firm keep its own baseline level of investment, then look at how changes in value and capital for that same firm over time relate to changes in its own investment.”  
- This is more realistic for corporate investment: GM and IBM don’t behave like generic clones; each has its own strategy that is stable over time.

So, **even if the pooled R² is higher, the FE results are the ones you trust for cause-and-effect statements**, because they control for unobserved firm characteristics that don’t change over time.[4][5]

## What the Hausman test tells you about Random vs Fixed Effects

Your Hausman test compares FE and RE:

- If your Hausman p‑value was **small (< 0.05)**, it means:  
  - The RE model and FE model give **meaningfully different coefficients** on value and capital.  
  - That is evidence that **unobserved firm traits are correlated with value and/or capital**.  
  - In plain English: “The kinds of firms that are more valuable or have more capital are also systematically different in ways we can’t see, and those hidden differences matter for investment.”  
  - Therefore, the **Random Effects assumption is violated**, and you should rely on **Fixed Effects**.

- If the Hausman p‑value were **large (≥ 0.05)**, you would say:  
  - “We don’t see a strong difference between FE and RE, so assuming RE is okay is not contradicted by the data.”  
  - In that case, you could use RE for efficiency.

Given a typical Grunfeld application, the data usually indicate that **FE is safer**, because firm-specific traits are indeed related to value and capital.[3][6][1]

## Overall layman summary

- The Grunfeld data show that big, valuable firms invest more, and firms with more capital also keep investing.  
- However, each firm has its own stable investment culture that you cannot see directly, and this culture matters a lot.  
- Because of that, models that ignore these firm differences (pooled, and in many cases random effects) are **too simplistic**, even if they produce a higher R².  
- The **Fixed Effects model** is the one that most honestly says:  
  “Given each firm’s own character, when its value and capital change, here is how its investment tends to change.”

[1](https://search.r-project.org/CRAN/refmans/AER/html/Grunfeld.html)
[2](https://www.statsmodels.org/dev/datasets/generated/grunfeld.html)
[3](https://www.stat.auckland.ac.nz/~wild/data/Rdatasets/doc/plm/Grunfeld.html)
[4](https://sites.google.com/site/econometricsacademy/masters-econometrics/panel-data-models)
[5](https://tilburgsciencehub.com/topics/analyze/causal-inference/panel-data/paneldata/)
[6](https://www.zeileis.org/papers/Kleiber+Zeileis-2010.pdf)